In [23]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import requests
import math

In [24]:
# Load and prepare data from the raw workbook
df = pd.read_excel('Data/UNHCR_Flow_Data.xlsx', sheet_name='DATA')
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Count"] = pd.to_numeric(df["Count"], errors="coerce")
df = df.dropna(subset=["Year", "Count"])

print(f"Dataset shape: {df.shape}")
print(f"Available years: {sorted(df['Year'].unique())}")

Dataset shape: (107341, 10)
Available years: [np.int64(1962), np.int64(1963), np.int64(1964), np.int64(1965), np.int64(1966), np.int64(1967), np.int64(1968), np.int64(1969), np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2

In [25]:
# Fetch country coordinates from REST Countries API
country_coords = {}
try:
    resp = requests.get("https://restcountries.com/v3.1/all?fields=cca3,latlng", timeout=10)
    resp.raise_for_status()
    for c in resp.json():
        iso3 = c.get("cca3")
        latlng = c.get("latlng")
        if iso3 and latlng:
            country_coords[iso3] = tuple(latlng)
except Exception as e:
    print(f"Warning: Could not fetch country coordinates: {e}")

print(f"Loaded coordinates for {len(country_coords)} countries")

Loaded coordinates for 250 countries


In [26]:
def curved_arc(lat1, lon1, lat2, lon2, n=50, height=0.12):
    """Interpolate a curved arc between two geographic points.
    height controls how much the arc bulges relative to the straight-line distance.
    """
    t = np.linspace(0, 1, n)
    lats = lat1 + t * (lat2 - lat1)
    lons = lon1 + t * (lon2 - lon1)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    dist = (dlat**2 + dlon**2) ** 0.5
    if dist < 0.01:
        return lats.tolist(), lons.tolist()
    # Perpendicular unit vector (90 deg CCW from travel direction)
    perp_lat, perp_lon = -dlon / dist, dlat / dist
    bulge = 4 * t * (1 - t) * height * dist
    return (lats + bulge * perp_lat).tolist(), (lons + bulge * perp_lon).tolist()


def calc_bearing(lat1, lon1, lat2, lon2):
    r = math.radians
    rl1, rl2 = r(lat1), r(lat2)
    dlon = r(lon2 - lon1)
    x = math.sin(dlon) * math.cos(rl2)
    y = math.cos(rl1) * math.sin(rl2) - math.sin(rl1) * math.cos(rl2) * math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

In [27]:
# Setup visualization parameters
N_BUCKETS = 5
N_TRACES = 2 + N_BUCKETS + 1

# Global log scale for consistent color reference across all maps
global_max_c = 1_000_000  # cap at 1M for consistent colour scale
log_max = float(np.log1p(global_max_c))
log_min = float(np.log1p(1_000))  # colours start at 1k
global_max_f = float(df["Count"].max())
global_log_max_f = float(np.log1p(global_max_f))

# Colorbar ticks
_tick_raw = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000, 5_000_000]
_tick_vals = [float(np.log1p(v)) for v in _tick_raw if v <= global_max_c]
_tick_text = ["1k", "5k", "10k", "50k", "100k", "500k", "1M", "5M"][:len(_tick_vals)]

# Arrow and line styling
FLOW_LINE_COLOR = "rgba(52, 152, 219, 0.55)"
ARROW_COLOR = "rgba(52, 152, 219, 0.85)"
BUCKET_WIDTHS = [0.9, 1.5, 2.2, 3.0, 4.0]

print(f"Max flow in dataset: {global_max_f:,.0f}")
print(f"Log scale range: {log_min:.2f} to {log_max:.2f}")

Max flow in dataset: 1,525,000
Log scale range: 6.91 to 13.82


In [28]:
def build_flow_map(origin_iso, start_year, end_year, highlight_color, title, output_path):
    """Generate animated choropleth map for refugee flows from a specific country."""
    sub = df[
        (df["OriginISO"] == origin_iso)
        & (df["Year"].between(start_year, end_year))
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows found for {origin_iso} between {start_year} and {end_year}")

    yearly_flows = sub.groupby(["Year", "OriginISO", "AsylumISO"], as_index=False)["Count"].sum()
    asylum_lookup = df[["AsylumISO", "AsylumName"]].drop_duplicates()
    origin_label = sub["OriginName"].dropna().iloc[0] if sub["OriginName"].dropna().any() else origin_iso
    years = sorted(yearly_flows["Year"].unique())
    n_traces_local = 2 + N_BUCKETS + 1

    def make_traces(year):
        yf = yearly_flows[yearly_flows["Year"] == year]

        dest_year = yf.groupby("AsylumISO", as_index=False)["Count"].sum()
        dest_year = dest_year.merge(asylum_lookup, on="AsylumISO", how="left")
        cd = np.column_stack([
            dest_year["AsylumISO"].values,
            dest_year["AsylumName"].values,
            [f"{int(x):,}" for x in dest_year["Count"].values],
        ])
        choropleth = go.Choropleth(
            locations=dest_year["AsylumISO"],
            z=np.log1p(dest_year["Count"].values),
            colorscale="Reds",
            zmin=log_min,
            zmax=log_max,
            customdata=cd,
            hovertemplate=(
                "<b>%{customdata[1]}</b><br>"
                "─────────────────────────<br>"
                f"Refugees received from {origin_label}: &nbsp;<b>%{{customdata[2]}}</b>"
                "<extra></extra>"
            ),
            colorbar=dict(
                title=f"Refugees<br>received from<br>{origin_label}",
                tickvals=_tick_vals,
                ticktext=_tick_text,
                x=1.01,
                thickness=14,
            ),
            showscale=True,
            marker_line_width=0.3,
            marker_line_color="white",
        )

        highlight = go.Choropleth(
            locations=[origin_iso],
            z=[1],
            colorscale=[[0, highlight_color], [1, highlight_color]],
            zmin=0,
            zmax=1,
            showscale=False,
            hovertemplate=f"<b>{origin_label}</b><extra></extra>",
            marker_line_width=1.8,
            marker_line_color="white",
        )

        bucket_lons = [[] for _ in range(N_BUCKETS)]
        bucket_lats = [[] for _ in range(N_BUCKETS)]
        a_lons, a_lats, a_angles, a_sizes, a_text = [], [], [], [], []

        for _, row in yf.iterrows():
            o = country_coords.get(row["OriginISO"])
            a = country_coords.get(row["AsylumISO"])
            if o is None or a is None:
                continue

            ratio = min(np.log1p(row["Count"]) / global_log_max_f, 1.0)
            bucket = min(int(ratio * N_BUCKETS), N_BUCKETS - 1)

            arc_lats, arc_lons = curved_arc(o[0], o[1], a[0], a[1])
            bucket_lons[bucket] += arc_lons + [None]
            bucket_lats[bucket] += arc_lats + [None]

            a_lons.append(a[1])
            a_lats.append(a[0])
            a_angles.append(calc_bearing(arc_lats[-2], arc_lons[-2], arc_lats[-1], arc_lons[-1]))
            a_sizes.append(7 + int(ratio * 12))
            a_text.append(
                f"<b>{row['OriginISO']} → {row['AsylumISO']}</b><br>"
                f"─────────────────────────<br>"
                f"People moving: &nbsp;<b>{int(row['Count']):,}</b>"
            )

        line_traces = []
        for b in range(N_BUCKETS):
            line_traces.append(go.Scattergeo(
                geo="geo",
                lon=bucket_lons[b] or [None],
                lat=bucket_lats[b] or [None],
                mode="lines",
                line=dict(width=BUCKET_WIDTHS[b], color=FLOW_LINE_COLOR),
                showlegend=False,
                hoverinfo="skip",
            ))

        arrows = go.Scattergeo(
            geo="geo",
            lon=a_lons or [None],
            lat=a_lats or [None],
            mode="markers",
            marker=dict(
                symbol="triangle-up",
                size=a_sizes or [0],
                color=ARROW_COLOR,
                angle=a_angles or [0],
                line=dict(width=0),
            ),
            text=a_text or [""],
            hovertemplate="%{text}<extra></extra>",
            showlegend=False,
        )

        return [choropleth, highlight] + line_traces + [arrows]

    fig = go.Figure(data=make_traces(years[0]))
    fig.frames = [
        go.Frame(data=make_traces(y), traces=list(range(n_traces_local)), name=str(y))
        for y in years
    ]

    fig.update_layout(
        title=title,
        geo=dict(
            showframe=False,
            showcoastlines=True,
            coastlinecolor="gray",
            bgcolor="#b3d9ff",
            projection_type="natural earth",
        ),
        hoverlabel=dict(bgcolor="white", font_size=13, font_family="Arial"),
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "y": 0.02,
            "x": 0.05,
            "xanchor": "right",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [None, {"frame": {"duration": 1800}, "fromcurrent": True,
                                  "transition": {"duration": 400}}]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}]
                },
            ],
        }],
        sliders=[{
            "active": 0,
            "steps": [
                {
                    "args": [[str(y)], {"frame": {"duration": 1800}, "mode": "immediate",
                                       "transition": {"duration": 300}}],
                    "label": str(y),
                    "method": "animate"
                }
                for y in years
            ],
            "x": 0.1,
            "len": 0.85,
            "y": 0,
            "currentvalue": {"prefix": "Year: ", "font": {"size": 14}, "xanchor": "center"},
            "pad": {"b": 10, "t": 50},
        }],
        margin=dict(l=0, r=0, t=50, b=80),
        height=600,
    )

    fig.write_html(output_path)
    return fig

In [29]:
# Generate maps for three conflict scenarios
afghanistan_fig = build_flow_map(
    "AFG",
    1980,
    1988,
    "rgba(241, 196, 15, 0.88)",
    "Afghanistan refugee flows 1980-1988",
    "../visualizations/afghanistan_flows.html",
)
afghanistan_fig.show()

rwanda_fig = build_flow_map(
    "RWA",
    1994,
    1996,
    "rgba(46, 204, 113, 0.88)",
    "Rwanda refugee flows 1994-1996",
    "../visualizations/rwanda_flows.html",
)
rwanda_fig.show()

ukraine_fig = build_flow_map(
    "UKR",
    2022,
    2025,
    "rgba(52, 152, 219, 0.88)",
    "Ukraine refugee flows 2022-2025",
    "../visualizations/ukraine_flows.html",
)
ukraine_fig.show()